# Imports

In [0]:
# init

from pyspark.sql import functions as F
from pyspark.sql.functions import when, col

# Read sales table from silver_schema

In [0]:
df_sales_details = spark.table("workspace.silver_schema.crm_sales_details")
df_sales_details.createOrReplaceTempView("sales")

df_dim_customer = spark.table("workspace.gold_schema.dim_customer")
df_dim_customer.createOrReplaceTempView("customer")

df_dim_product = spark.table("workspace.gold_schema.dim_product")
df_dim_product.createOrReplaceTempView('product')

In [0]:
display(df_dim_product)
display(df_sales_details)

In [0]:
for col in df_sales_details.columns:
    print (f"s.{col},")

In [0]:
query = """
    SELECT 
        /* Only include necessary columne and Rename columns to more friendly names*/

        s.sales_order_number AS order_number,

        -- s.sales_product_key, # Replace this with newly derived key after joining tables
        c.customer_key,

        -- s.sales_customer_id, # Replace this with newly derived key after joining tables
        p.product_key,

        s.sales_order_date AS order_date,
        s.sales_shipping_date AS shipping_date,
        s.sales_due_date As due_date,
        s.sales_sales AS sales_amount,
        s.sales_quantity AS quantity,
        s.sales_price AS price

    FROM sales s
    
    INNER JOIN customer c
        ON s.sales_customer_id = c.customer_id
   
    INNER JOIN product p
        ON s.sales_product_key = p.product_number
  
    -- WHERE c.customer_id IS NULL OR p.product_number IS NULL -- Conditional to check if all keys match (Initially used with LEFT JOIN)

"""


df_fact_sales = spark.sql(query)
df_fact_sales.display()

# Write fact_sales into the gold_schema

In [0]:
df_fact_sales.write.mode('overwrite').format('delta').saveAsTable("workspace.gold_schema.fact_sales")

# Read from fact-sales from gold_schema

In [0]:
df_gold_fact_sales = spark.table("workspace.gold_schema.fact_sales")
df_gold_fact_sales.display()